# 🩻 Qure.ai — AI-Assisted TB Screening Prototype
## Complete Full-Code ML Pipeline (Educational Demonstration)

**Course:** Introduction to AI & ML | BBA AI/ML | Chitkara Business School  
**Organization:** Qure.ai  
**Business Problem:** Support faster screening and prioritization of chest X-rays for possible tuberculosis (TB), while keeping clinicians responsible for diagnosis and treatment.

> **Important:** This notebook is an educational prototype. It does **not** reproduce Qure.ai's proprietary qXR model and does not diagnose TB. Because no real chest X-ray dataset was provided, the model below uses a **synthetic tabular dataset** representing screening/workflow variables. It demonstrates the ML pipeline, not clinical performance.

### Pipeline followed
1. Data Collection → 2. Data Understanding → 3. Data Cleaning → 4. Outlier Detection & Treatment → 5. Feature Selection → 6. Define Target Variable → 7. Encode Target Variable → 8. Train-Test Split → 9. Feature Standardisation → 10. Model Building → 11. Model Training → 12. Prediction → 13. Model Evaluation → 14. Model Interpretation → 15. Final Output


---
## Step 1 — Data Collection

**What:** Create/load a dataset representing screening cases.  
**Why:** A supervised ML model needs input features and a target label.

Since real patient X-rays are not supplied, we create a **synthetic dataset** for demonstration. In a real deployment, image pixels, expert labels, confirmatory results, and relevant clinical/workflow data would require secure governance and validation.


In [ ]:
# STEP 1: DATA COLLECTION
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducible synthetic dataset
np.random.seed(42)
n = 300

df = pd.DataFrame({
    "Case_ID": [f"XR{i:04d}" for i in range(1, n + 1)],
    "Age": np.random.randint(18, 81, n),
    "Cough_Days": np.random.poisson(10, n).clip(0, 45),
    "Weight_Loss": np.random.binomial(1, 0.28, n),
    "Fever": np.random.binomial(1, 0.35, n),
    "Prior_TB": np.random.binomial(1, 0.12, n),
    "Xray_Abnormality_Score": np.clip(np.random.normal(0.45, 0.22, n), 0, 1),
    "Referral_Priority": np.random.binomial(1, 0.30, n)
})

# Synthetic target for demonstration only:
# higher symptoms + abnormality score + prior TB -> greater probability of TB-positive screening label
logit = (
    -4.0
    + 0.055 * df["Age"]
    + 0.075 * df["Cough_Days"]
    + 0.85 * df["Weight_Loss"]
    + 0.65 * df["Fever"]
    + 1.00 * df["Prior_TB"]
    + 3.00 * df["Xray_Abnormality_Score"]
)
prob = 1 / (1 + np.exp(-logit))
df["TB_Screen_Positive"] = np.random.binomial(1, prob)

# Add a few intentionally messy values for cleaning practice
df.loc[5, "Age"] = np.nan
df.loc[12, "Cough_Days"] = -3
df.loc[25, "Xray_Abnormality_Score"] = 1.8
df.loc[40, "Cough_Days"] = np.nan

df.head()


**How to read the output:**  
Each row represents a screening case. The target `TB_Screen_Positive` is the label the model will learn to predict. The abnormality score is a synthetic proxy for image-derived information; it is **not** a real Qure.ai/qXR score.


---
## Step 2 — Data Understanding / Data Inspection

**What:** Inspect rows, columns, data types, missing values, duplicates and statistics.  
**Why:** We need to understand data quality before cleaning or modelling.


In [ ]:
# STEP 2: DATA UNDERSTANDING
print("Shape (rows, columns):", df.shape)

print("\nColumn data types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nStatistical summary:")
display(df.describe(include="all"))


---
## Step 3 — Data Cleaning

**What:** Fix missing and invalid values.  
**Why:** Invalid values can create misleading patterns and reduce model reliability.

We will:
- fill numeric missing values with the median;
- replace impossible negative cough duration;
- handle an abnormality score outside its valid 0–1 range;
- remove duplicate rows.


In [ ]:
# STEP 3: DATA CLEANING
print("Rows before cleaning:", len(df))

before = len(df)
df = df.drop_duplicates()
print("Duplicate rows removed:", before - len(df))

# Invalid cough duration cannot be negative
invalid_cough = (df["Cough_Days"] < 0).sum()
df.loc[df["Cough_Days"] < 0, "Cough_Days"] = np.nan
print("Invalid negative Cough_Days values:", invalid_cough)

# Abnormality score is defined on 0–1 in this synthetic example
invalid_score = ((df["Xray_Abnormality_Score"] < 0) | (df["Xray_Abnormality_Score"] > 1)).sum()
df.loc[(df["Xray_Abnormality_Score"] < 0) | (df["Xray_Abnormality_Score"] > 1),
       "Xray_Abnormality_Score"] = np.nan
print("Invalid abnormality scores:", invalid_score)

# Fill missing numeric values with median
for col in ["Age", "Cough_Days", "Xray_Abnormality_Score"]:
    missing = df[col].isna().sum()
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    print(f"{col}: filled {missing} missing value(s) with median = {median_value:.3f}")

print("\nRemaining missing values:")
print(df.isna().sum())


---
## Step 4 — Outlier Detection & Treatment

**What:** Detect extreme values using the IQR method and cap them.  
**Why:** Extreme values can distort some ML models and statistics.

For this educational dataset, we cap numeric feature outliers rather than deleting complete cases.


In [ ]:
# STEP 4: OUTLIER DETECTION & TREATMENT
numeric_cols = ["Age", "Cough_Days", "Xray_Abnormality_Score", "Referral_Priority"]

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower, upper)
    print(f"{col}: {count} outlier(s) capped | range = [{lower:.3f}, {upper:.3f}]")

print("\nSummary after outlier treatment:")
display(df[numeric_cols].describe())


---
## Step 5 — Feature Selection

**What:** Select variables that are relevant to the screening task.  
**Why:** We should avoid using identifiers or information that would not be available at prediction time.

`Case_ID` is removed because it is only an identifier. The remaining variables represent synthetic screening/clinical features.


In [ ]:
# STEP 5: FEATURE SELECTION
features = [
    "Age",
    "Cough_Days",
    "Weight_Loss",
    "Fever",
    "Prior_TB",
    "Xray_Abnormality_Score",
    "Referral_Priority"
]

X = df[features].copy()
print("Selected features:")
print(features)


---
## Step 6 — Define Target Variable

The target is `TB_Screen_Positive`.

**Target meaning:** 1 = synthetic screening-positive case; 0 = synthetic screening-negative case.

This label is for demonstration only and must not be interpreted as a medical diagnosis.


In [ ]:
# STEP 6: DEFINE TARGET
y = df["TB_Screen_Positive"].copy()

print("Target distribution:")
print(y.value_counts())
print("\nTarget proportions:")
print(y.value_counts(normalize=True).round(3))


---
## Step 7 — Encode Target Variable

The target is already binary:
- `0` = negative
- `1` = positive

No additional encoding is required.


In [ ]:
# STEP 7: TARGET ENCODING
y = y.astype(int)
print("Unique encoded target values:", sorted(y.unique()))


---
## Step 8 — Train-Test Split

**What:** Split data into training and testing sets.  
**Why:** The model should be evaluated on data it did not train on.

We use an 80:20 split and stratify by the target to preserve class proportions.


In [ ]:
# STEP 8: TRAIN-TEST SPLIT
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


---
## Step 9 — Feature Standardisation

**What:** Put numeric features on comparable scales.  
**Why:** Logistic Regression works better when continuous variables are on similar scales.

`StandardScaler` is fitted only on training data to avoid data leakage.


In [ ]:
# STEP 9: FEATURE STANDARDISATION
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data standardised successfully.")


---
## Step 10 — Model Building

We use **Logistic Regression** as a simple, interpretable binary-classification model.

It estimates the probability of a case belonging to the positive screening class.


In [ ]:
# STEP 10: MODEL BUILDING
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
print(model)


---
## Step 11 — Model Training

The model learns relationships between the selected features and the synthetic target using the training data.


In [ ]:
# STEP 11: MODEL TRAINING
model.fit(X_train_scaled, y_train)
print("Model training completed.")


---
## Step 12 — Prediction

We generate:
1. a predicted class (0 or 1);
2. a predicted probability for the positive class.

In a real clinical system, a probability/score would need clinically validated thresholds and workflow design.


In [ ]:
# STEP 12: PREDICTION
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

prediction_output = X_test.copy()
prediction_output["Actual"] = y_test.values
prediction_output["Predicted"] = y_pred
prediction_output["Positive_Probability"] = y_prob.round(3)

display(prediction_output.head(10))


---
## Step 13 — Model Evaluation

We evaluate using:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion matrix

For a screening-oriented application, sensitivity/recall is especially important to examine, but no single metric is sufficient for clinical deployment.


In [ ]:
# STEP 13: MODEL EVALUATION
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1-score : {f1:.3f}")
print(f"ROC-AUC  : {auc:.3f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted 0", "Predicted 1"],
            yticklabels=["Actual 0", "Actual 1"])
plt.title("Confusion Matrix — Educational Prototype")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.show()


---
## Step 14 — Model Interpretation

Logistic Regression provides coefficients that indicate the direction and relative contribution of each feature in this synthetic model.

**Important:** These coefficients are properties of this educational dataset. They are **not** evidence about clinical risk factors and should not be used for medical decisions.


In [ ]:
# STEP 14: MODEL INTERPRETATION
coef_table = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", ascending=False)

display(coef_table)

plt.figure(figsize=(9, 5))
plt.barh(coef_table["Feature"], coef_table["Coefficient"])
plt.axvline(0, linewidth=1)
plt.title("Logistic Regression Coefficients — Synthetic Data")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.show()


---
## Step 15 — Final Output

The model can produce a **screening-support prediction** for new cases in this prototype.

A real Qure.ai-style workflow is better represented as:

**Chest X-ray → AI image analysis → screening/priority output → clinician review → confirmatory testing/action**

The AI output should support clinical workflow rather than replace professional diagnosis or treatment decisions.


In [ ]:
# STEP 15: FINAL OUTPUT
final_output = prediction_output.copy()

final_output["Screening_Flag"] = np.where(
    final_output["Predicted"] == 1,
    "Priority for clinical review",
    "Lower AI screening priority"
)

display(final_output.head(15))

print("\nPipeline completed successfully.")
print("This is an educational prototype using synthetic data, not a clinical diagnostic model.")


---
# Business Interpretation

### How this connects to the Qure.ai case study

- **Business problem:** Large volumes of chest X-rays can create screening workload and delays, especially where specialist resources are limited.
- **AI/ML approach:** Computer vision and deep learning can analyze chest X-ray images and produce screening or prioritization outputs.
- **Prototype shown here:** Logistic Regression demonstrates the general supervised-classification workflow using synthetic tabular features.
- **Business value:** Potentially faster screening, improved workflow organization, better use of specialist time, and support for TB case-finding.
- **Human oversight:** Doctors/radiologists remain responsible for clinical decisions.

### Limitations

1. This notebook does not use real chest X-ray images.
2. The dataset and target labels are synthetic.
3. The model is not Qure.ai's proprietary model.
4. Performance numbers from this notebook cannot be used to claim clinical effectiveness.
5. Real deployment requires representative clinical data, privacy/security controls, external validation, monitoring, and regulatory/clinical governance.


---
# Viva / Presentation Points

**Why Computer Vision?**  
Chest X-rays are image data, so computer vision is suitable for extracting visual patterns.

**Why Machine Learning?**  
ML can learn patterns from labelled examples and generate predictions or screening-support scores.

**Why Logistic Regression in this notebook?**  
It is simple, fast and relatively interpretable for demonstrating binary classification. A production chest-X-ray system would generally require image-based deep-learning methods and rigorous clinical validation.

**What is the target variable?**  
`TB_Screen_Positive`, a synthetic binary label used only for this demonstration.

**What is the business output?**  
A screening-support prediction/priority flag that can help organize cases for clinical review.

**Does the AI diagnose TB?**  
No. In the case-study workflow, AI supports screening and prioritization; clinicians make the final medical decisions.
